Este código automatiza la carga de artefactos de aprendizaje automático previamente entrenados para utilizarlos en una etapa de producción o inferencia, asegurando que no se realice ningún proceso de reentrenamiento.

### **Importación de librerías**
Carga `joblib` para abrir los archivos serializados que contienen el modelo y sus configuraciones, y `pandas` para gestionar la estructura de los datos que se procesarán.

### **Carga del pipeline principal**
Recupera el objeto `pipeline_v3`, el cual contiene todo el flujo de trabajo final: desde la transformación técnica de los datos de entrada hasta el algoritmo de predicción ya entrenado.

### **Carga del transformador de etiquetas**
Obtiene el codificador `le` (`LabelEncoder`), utilizado para convertir las predicciones numéricas del modelo de vuelta a sus categorías o nombres de texto originales.

### **Carga de la estructura de datos**
Extrae la lista `columnas_finales_v3`, que define el orden exacto y la cantidad de variables que el modelo requiere para recibir datos nuevos de forma correcta.

### **Verificación en consola**
Imprime un mensaje informativo confirmando la lectura exitosa del pipeline, el número total de columnas esperadas y la lista de las clases que el modelo es capaz de predecir.


In [13]:
import joblib
import pandas as pd

# Carga de artefactos ya entrenados (Capítulo 18 — nunca se reentrena aquí)
pipeline_v3 = joblib.load(r"..\models\modelo_perfil_energetico_final_v3.joblib")
le = joblib.load(r"..\models\label_encoder_v3.joblib")
columnas_finales_v3 = joblib.load(r"..\models\columnas_requeridas_final_v3.joblib")

print(f"Pipeline cargado. Columnas esperadas: {len(columnas_finales_v3)}")
print(f"Clases: {list(le.classes_)}")

Pipeline cargado. Columnas esperadas: 55
Clases: ['Eficiente', 'Ineficiente', 'Moderado']


Este código define un componente de ingeniería de características personalizado y lo integra en un pipeline de Scikit-Learn para procesar las variables del modelo sin necesidad de reentrenamiento.

### **Función de división segura**
Evita errores de división por cero (`ZeroDivisionError`) o valores nulos convirtiendo los datos a numéricos, aislando los denominadores en cero con una máscara lógica y reemplazando cualquier valor infinito (`inf`) resultante por un valor por defecto seguro.

### **Clase FeatureEngineerV3**
Hereda de `BaseEstimator` y `TransformerMixin` para actuar como un transformador oficial dentro del pipeline. Traduce un conjunto de variables de entrada básicas (12 columnas) en una estructura expandida más compleja (55 columnas) requerida por el modelo.

### **Procesamiento de datos internos**
El método `transform()` limpia las variables numéricas rellenando valores nulos, extrae variables categóricas mediante codificación One-Hot manual (Dummies), calcula indicadores de estacionalidad con funciones seno y coseno basadas en el mes actual del servidor, y genera multiplicadores según la calidad del aislamiento térmico.

### **Cálculo de nuevas características**
Combina los datos crudos para fabricar métricas avanzadas sobre densidad habitacional, obsolescencia o antigüedad cruzada del inmueble y los electrodomésticos, proporciones de iluminación LED, y el consumo histórico cruzado con el tipo de propiedad.

### **Alineación de columnas**
Filtra y ordena el conjunto de datos final de salida utilizando estrictamente la lista almacenada en `self.columnas_finales` para garantizar que la estructura resultante coincida de manera exacta con las expectativas de las capas posteriores del modelo.

### **Ensamblaje del pipeline completo**
Extrae los componentes de preprocesamiento y clasificación del objeto `pipeline_v3` original y construye un nuevo objeto `Pipeline` secuencial que automatiza en un solo paso la ingeniería de variables, el preprocesamiento técnico y la inferencia final.


In [14]:
import sys
import os

sys.path.append("..")  # raíz del proyecto, para poder importar desde src/

from sklearn.pipeline import Pipeline
from src.features.feature_engineer_v3 import FeatureEngineerV3, division_segura

clasificador = pipeline_v3.named_steps["classifier"]
preprocessor_v3 = pipeline_v3.named_steps["preprocessor"]

pipeline_v3_completo = Pipeline([
    ("feature_engineering", FeatureEngineerV3(columnas_finales=columnas_finales_v3)),
    ("preprocessor", preprocessor_v3),
    ("classifier", clasificador),
])

# Smoke test autosuficiente: un registro de ejemplo escrito a mano, no depende de X_test_crudo
registro_prueba = pd.DataFrame([{
    "tipo_inmueble": "Casa Unifamiliar",
    "superficie_m2": 120.5,
    "num_personas": 4,
    "cantidad_equipos_total": 12,
    "horas_uso_aa_dia": 6.5,
    "consumo_kwh_mensual": 450.0,
    "consumo_kwh_mes_anterior": 430.0,
    "aislamiento_termico": "Regular",
    "pct_iluminacion_led": 65.0,
    "antiguedad_construccion_anios": 15,
    "zona": "Urbana Interior",
    "antiguedad_electrodomesticos_anios": 8,
}])

prueba = pipeline_v3_completo.predict_proba(registro_prueba)
print("Smoke test OK:", prueba)

Smoke test OK: [[0.0731256  0.01628758 0.91058682]]


Este código realiza la validación técnica del nuevo pipeline completo, reconstruyendo un conjunto de datos crudos a partir del conjunto de prueba original para comparar las predicciones paso a paso y garantizar que los resultados matemáticos sean idénticos.

### **Carga y partición de datos estructurados**
Importa el dataset procesado y aplica una segmentación estratificada con `train_test_split` (usando un 20% para pruebas) garantizando que la distribución de la variable objetivo codificada se mantenga idéntica en el conjunto de entrenamiento y el de validación.

### **Función de ingeniería inversa categórica**
Define la función auxiliar `onehot_a_categoria`, encargada de realizar una conversión inversa de las variables One-Hot (Dummies binarias). Examina múltiples columnas e identifica cuál contiene el valor activo para consolidar la etiqueta de texto original en una sola serie.

### **Reconstrucción del conjunto de pruebas crudo**
Extrae los índices del subconjunto de prueba para recuperar su estado original, mapea de vuelta las categorías de tipo de inmueble, aislamiento y zona, e incorpora una columna temporal de control del mes del año con el fin de evitar discrepancias debido al reloj interno del servidor.

### **Auditoría de integridad de datos**
Filtra el parámetro de fecha de control y evalúa el DataFrame crudo recién construido mediante una sumatoria total de nulos; esto actúa como una compuerta de calidad que valida que la reconstrucción inversa no generó registros incompletos o vacíos.

### **Prueba de consistencia y equivalencia probabilística**
Calcula y contrasta las matrices de probabilidad de predicción (`predict_proba`) procesando los datos estructurados en el modelo antiguo y los datos crudos en el nuevo pipeline integrado, calculando la desviación absoluta máxima entre ambos resultados.

### **Diagnóstico de certificación del pipeline**
Mide la tasa exacta de coincidencia entre las clases predichas y emite un mensaje de éxito si las desviaciones de probabilidad caen por debajo del umbral de tolerancia estricto, o despliega una alerta de discrepancia aislando los primeros cinco registros problemáticos si se detecta cualquier variación.


In [15]:
from sklearn.model_selection import train_test_split
import warnings
import numpy as np
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
ruta_dataset = r"..\datasets\processed\03_feature_engineering.csv"
target = "perfil_energetico"

df = pd.read_csv(ruta_dataset, sep=",", encoding="utf-8-sig")

X = df.drop(columns=["id_registro", target])
y_encoded = le.transform(df[target])
X_v3 = X[columnas_finales_v3]

X_train_v3, X_test_v3, y_train_v3, y_test_v3 = train_test_split(
    X_v3, y_encoded, test_size=0.2, stratify=y_encoded, random_state=RANDOM_STATE
)
print("Test set reconstruido:", X_test_v3.shape)

# --- Reconstruir los 12 campos crudos para las mismas filas del test set ---
def onehot_a_categoria(df_fuente, prefijo, categorias):
    resultado = pd.Series(index=df_fuente.index, dtype=object)
    for cat in categorias:
        resultado.loc[df_fuente[f"{prefijo}_{cat}"] == 1] = cat
    return resultado

df_test_original = df.loc[X_test_v3.index]

tipo_inmueble_cat = onehot_a_categoria(df_test_original, "tipo_inmueble",
    ["Apartamento", "Casa Unifamiliar", "Pequeño Establecimiento Comercial"])
aislamiento_cat = onehot_a_categoria(df_test_original, "aislamiento_termico", ["Bueno", "Malo", "Regular"])
zona_cat = onehot_a_categoria(df_test_original, "zona", ["Suburbana", "Urbana Costera", "Urbana Interior"])

X_test_crudo = pd.DataFrame({
    "tipo_inmueble": tipo_inmueble_cat,
    "superficie_m2": df_test_original["superficie_m2"],
    "num_personas": df_test_original["num_personas"],
    "cantidad_equipos_total": df_test_original["cantidad_equipos_total"],
    "horas_uso_aa_dia": df_test_original["horas_uso_aa_dia"],
    "consumo_kwh_mensual": df_test_original["consumo_kwh_mensual"],
    "consumo_kwh_mes_anterior": df_test_original["consumo_kwh_mes_anterior"],
    "aislamiento_termico": aislamiento_cat,
    "pct_iluminacion_led": df_test_original["pct_iluminacion_led"],
    "antiguedad_construccion_anios": df_test_original["antiguedad_construccion_anios"],
    "zona": zona_cat,
    "antiguedad_electrodomesticos_anios": df_test_original["antiguedad_electrodomesticos_anios"],
    "_mes_numero_test": df_test_original["mes_numero"],  # solo para esta prueba
}, index=X_test_v3.index)

print("Set crudo de validación:", X_test_crudo.shape)
print(X_test_crudo.drop(columns=["_mes_numero_test"]).isna().sum().sum(), "valores nulos (debe ser 0)")

# --- Comparación ---
probas_original = pipeline_v3.predict_proba(X_test_v3)
probas_completo = pipeline_v3_completo.predict_proba(X_test_crudo)

diferencia_max = np.abs(probas_original - probas_completo).max()
coincidencias = (probas_original.argmax(axis=1) == probas_completo.argmax(axis=1)).mean()

print(f"\nDiferencia máxima absoluta entre probabilidades: {diferencia_max:.10f}")
print(f"Predicciones idénticas (misma clase): {coincidencias * 100:.4f}%")

if diferencia_max < 1e-6 and coincidencias == 1.0:
    print("\nVALIDACIÓN EXITOSA: el pipeline completo reproduce exactamente al pipeline v3 original.")
else:
    print("\nDIFERENCIAS DETECTADAS — no continuar hasta resolver esto.")
    idx_dif = np.where(probas_original.argmax(axis=1) != probas_completo.argmax(axis=1))[0]
    print(f"Filas con predicción distinta: {len(idx_dif)}")
    if len(idx_dif) > 0:
        print(X_test_crudo.iloc[idx_dif[:5]])

Test set reconstruido: (18908, 55)
Set crudo de validación: (18908, 13)
0 valores nulos (debe ser 0)

Diferencia máxima absoluta entre probabilidades: 0.0141921276
Predicciones idénticas (misma clase): 100.0000%

DIFERENCIAS DETECTADAS — no continuar hasta resolver esto.
Filas con predicción distinta: 0


Este código realiza un diagnóstico granular a nivel de variables, aislando exclusivamente el transformador de ingeniería de características para auditar y comparar cada una de las 55 columnas generadas contra los valores históricos originales del conjunto de datos.

### **Aislamiento del paso de ingeniería de variables**
Invoca de forma directa el método `transform()` del componente específico `feature_engineering` alojado dentro del pipeline unificado, procesando el DataFrame crudo para dar origen a la matriz expandida sin ejecutar los pasos posteriores de escalado o predicción.

### **Cálculo matricial de discrepancias matemáticas**
Efectúa una resta matricial directa convirtiendo ambos conjuntos de datos en arreglos de punto flotante de NumPy (`.values.astype(float)`), calculando posteriormente el valor absoluto del residuo para cuantificar cualquier desviación numérica o error de redondeo.

### **Construcción del reporte de auditoría**
Estructura un nuevo DataFrame de control que vincula cada etiqueta de la lista oficial de variables con su desviación máxima absoluta registrada y el volumen exacto de registros que superan una tolerancia matemática estricta definida en un millonésimo.

### **Ordenamiento y jerarquización de fallos**
Clasifica de manera descendente el listado de resultados basándose en la magnitud del error, permitiendo identificar de forma inmediata qué fórmulas o transformaciones específicas están sufriendo alteraciones o problemas de consistencia lógica.

### **Despliegue del top de variables críticas**
Formatea y muestra en la terminal las primeras diez filas del resumen omitiendo el índice automático del DataFrame, facilitando un diagnóstico visual rápido enfocado únicamente en las variables con mayores desajustes en el entorno de pruebas.
****

In [16]:
# Generar las 55 columnas usando SOLO el paso de feature engineering (sin pasar por preprocessor/classifier)
X_test_completo_generado = pipeline_v3_completo.named_steps["feature_engineering"].transform(X_test_crudo)

# Comparar columna por columna contra los valores originales del dataset
diferencias = (X_test_completo_generado.values.astype(float) - X_test_v3.values.astype(float))
diff_abs = np.abs(diferencias)

resumen_diferencias = pd.DataFrame({
    "columna": columnas_finales_v3,
    "diferencia_maxima": diff_abs.max(axis=0),
    "cantidad_filas_afectadas": (diff_abs > 1e-6).sum(axis=0)
}).sort_values("diferencia_maxima", ascending=False)

print(resumen_diferencias.head(10).to_string(index=False))

                           columna  diferencia_maxima  cantidad_filas_afectadas
       consumo_anterior_estacional       9.094947e-13                         0
      consumo_anterior_por_persona       2.273737e-13                         0
             superficie_por_equipo       1.136868e-13                         0
            superficie_por_persona       5.684342e-14                         0
       consumo_anterior_por_equipo       5.684342e-14                         0
           consumo_anterior_por_m2       1.421085e-14                         0
carga_equipos_antiguos_por_persona       1.421085e-14                         0
  indice_ineficiencia_constructiva       7.105427e-15                         0
               personas_por_equipo       1.776357e-15                         0
              horas_aa_por_persona       8.881784e-16                         0


Este código implementa un análisis estadístico de diagnóstico avanzado enfocado en rastrear y aislar las discrepancias de predicción por cada registro, identificando con precisión quirúrgica cuáles variables son las causantes de las diferencias en el peor caso detectado.

### **Cálculo de desviaciones probabilísticas por registro**
Determina la desviación absoluta máxima entre las predicciones del modelo original y el nuevo pipeline para cada fila de datos, reduciendo la matriz tridimensional de clases al peor margen de error individual mediante la función `max(axis=1)`.

### **Auditoría estadística de la distribución**
Convierte el vector de diferencias en una serie de Pandas y ejecuta el método `describe()` para arrojar métricas de dispersión clave (media, desviación estándar, percentiles y valor máximo), ofreciendo un panorama del comportamiento general del error.

### **Segmentación por umbral de tolerancia**
Aplica un filtro lógico con `np.where` basado en una cota fija de un milésimo para contabilizar el volumen exacto de registros que exhiben una inconsistencia numéricamente significativa en comparación con el tamaño total de la muestra de prueba.

### **Aislamiento del peor registro evaluado**
Utiliza `argmax()` sobre el vector de errores para extraer de forma automática el índice de la fila con el peor desempeño del conjunto, imprimiendo en consola la comparativa directa entre los vectores de probabilidad originales y reconstruidos para ese caso.

### **Análisis de origen de error por variable**
Construye un DataFrame de diagnóstico cruzado para la fila crítica que vincula los nombres de las 55 columnas con sus valores en ambos entornos, calculando el residuo matemático neto entre el valor original y el generado artificialmente.

### **Filtrado de variables causantes**
Remueve todas las columnas cuyo residuo sea exactamente cero mediante un filtro booleano, imprimiendo un listado depurado que expone únicamente las características específicas que sufrieron alteraciones en ese registro para facilitar la corrección del código.


In [17]:
diferencias_probas = np.abs(probas_original - probas_completo).max(axis=1)  # peor diferencia por fila, entre las 3 clases

print("Distribución de diferencias de probabilidad por fila:")
print(pd.Series(diferencias_probas).describe())

umbral_relevante = 0.001
filas_relevantes = np.where(diferencias_probas > umbral_relevante)[0]
print(f"\nFilas con diferencia > {umbral_relevante}: {len(filas_relevantes)} de {len(diferencias_probas)}")

if len(filas_relevantes) > 0:
    idx_peor = diferencias_probas.argmax()
    print(f"\nPeor caso -> posición #{idx_peor} en el test set")
    print("Probas originales:   ", probas_original[idx_peor])
    print("Probas reconstruidas:", probas_completo[idx_peor])

    print("\nColumnas donde esa fila específica difiere (aunque sea mínimamente):")
    comparacion_fila = pd.DataFrame({
        "columna": columnas_finales_v3,
        "original": X_test_v3.iloc[idx_peor].values,
        "generado": X_test_completo_generado.iloc[idx_peor].values,
    })
    comparacion_fila["diferencia"] = comparacion_fila["original"] - comparacion_fila["generado"]
    print(comparacion_fila[comparacion_fila["diferencia"] != 0].to_string(index=False))

Distribución de diferencias de probabilidad por fila:
count    1.890800e+04
mean     7.505885e-07
std      1.032107e-04
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.419213e-02
dtype: float64

Filas con diferencia > 0.001: 1 de 18908

Peor caso -> posición #15246 en el test set
Probas originales:    [0.68480789 0.01423736 0.30095475]
Probas reconstruidas: [0.67061577 0.01487842 0.31450581]

Columnas donde esa fila específica difiere (aunque sea mínimamente):
                    columna  original  generado    diferencia
consumo_anterior_por_equipo 25.840000 25.840000  3.552714e-15
      densidad_habitacional  0.037383  0.037383 -5.551115e-17
             equipos_por_m2  0.186916  0.186916 -8.326673e-17


Este código automatiza la fase final de despliegue del proyecto, exportando el pipeline consolidado en formato binario y generando tres archivos JSON complementarios que documentan la gobernanza del modelo, sus hiperparámetros de entrenamiento y la auditoría de fidelidad matemática.

### **Persistencia binaria del pipeline completo**
Captura la marca de tiempo actual del sistema e importa las dependencias de empaquetado para exportar de forma serializada con `joblib.dump` el objeto del pipeline unificado, dejándolo listo para su consumo directo en entornos de producción.

### **Generación de metadatos de gobernanza**
Construye el archivo `metadata_v3.json` que registra información estática crucial para el control de versiones, incluyendo las clases del codificador, la volumetría de variables de entrada, las métricas oficiales de rendimiento de la prueba y las versiones exactas de Scikit-Learn y LightGBM utilizadas.

### **Documentación de la configuración de entrenamiento**
Escribe el archivo `training_config_v3.json` capturando la procedencia de los datos, la estrategia de validación cruzada y los hiperparámetros de calibración activos del `LGBMClassifier` mediante el método `.get_params()`, usando el serializador `default=str` para manejar tipos de datos complejos.

### **Registro y bitácora de auditoría de fidelidad**
Crea el archivo `export_log_v3.json` donde almacena los resultados cuantitativos de la prueba de consistencia previa, documentando métricas estadísticas como la mediana y el error máximo absoluto junto con una conclusión técnica descriptiva sobre el comportamiento del punto flotante en el peor escenario.

### **Declaración de limitaciones del sistema**
Incorpora dentro del reporte de auditoría una sección descriptiva de riesgos conocidos, especificando la ausencia de datos sobre generación solar fotovoltaica en el modelo y advirtiendo sobre la dependencia del reloj del servidor para la estimación estacional del mes de consumo.

### **Confirmación y trazabilidad en consola**
Finaliza la ejecución imprimiendo un resumen formateado de los cuatro artefactos almacenados en la carpeta de destino, sirviendo como una lista de verificación visual que confirma el fin exitoso del flujo de integración y empaquetado del software.


In [18]:
import json
import sklearn
import lightgbm
from datetime import datetime

VERSION = "v3"
FECHA_EXPORTACION = datetime.now().isoformat()

# --- 1. Pipeline oficial ---
joblib.dump(pipeline_v3_completo, rf"..\models\model_pipeline_{VERSION}.joblib")

# --- 2. metadata_v3.json ---
metadata = {
    "version_modelo": VERSION,
    "algoritmo": "LGBMClassifier",
    "fecha_exportacion": FECHA_EXPORTACION,
    "clases": list(le.classes_),
    "cantidad_features_entrada": len(columnas_finales_v3),
    "cantidad_features_crudas_requeridas": 12,
    "columnas_requeridas_modelo": columnas_finales_v3,
    "metricas_test": {
        "accuracy": 0.91,
        "f1_macro": 0.91,
        "f1_por_clase": {"Eficiente": 0.93, "Ineficiente": 0.94, "Moderado": 0.88}
    },
    "librerias": {
        "scikit-learn": sklearn.__version__,
        "lightgbm": lightgbm.__version__
    }
}
with open(rf"..\models\metadata_{VERSION}.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# --- 3. training_config_v3.json ---
training_config = {
    "random_state": 42,
    "validacion": "Stratified K-Fold, 5 folds",
    "train_test_split": {"test_size": 0.2, "stratify": True, "random_state": 42},
    "dataset_origen": "03_feature_engineering.csv",
    "hiperparametros_lgbm": clasificador.get_params()
}
with open(rf"..\models\training_config_{VERSION}.json", "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2, default=str, ensure_ascii=False)

# --- 4. export_log_v3.json ---
export_log = {
    "fecha_exportacion": FECHA_EXPORTACION,
    "pipeline_incluye_feature_engineering": True,
    "validacion_fidelidad": {
        "filas_test_comparadas": int(len(diferencias_probas)),
        "diferencia_mediana": float(np.median(diferencias_probas)),
        "diferencia_maxima": float(diferencias_probas.max()),
        "filas_con_diferencia_mayor_a_0.001": int(len(filas_relevantes)),
        "conclusion": "Diferencia aislada a 1 fila de 18,908, causada por ruido de punto flotante (~1e-15) amplificado por un umbral de decisión del árbol. Sin sesgo sistemático (mediana=0). No representa un error de lógica en el feature engineering."
    },
    "limitaciones_conocidas": [
        "Dominio de generación solar excluido del modelo (no capturable desde el formulario)",
        "mes_numero se deriva de la fecha del servidor en producción, no de un campo del formulario"
    ]
}
with open(rf"..\models\export_log_{VERSION}.json", "w", encoding="utf-8") as f:
    json.dump(export_log, f, indent=2, ensure_ascii=False)

print("4 artefactos exportados a models/:")
print(f"  - model_pipeline_{VERSION}.joblib")
print(f"  - metadata_{VERSION}.json")
print(f"  - training_config_{VERSION}.json")
print(f"  - export_log_{VERSION}.json")

4 artefactos exportados a models/:
  - model_pipeline_v3.joblib
  - metadata_v3.json
  - training_config_v3.json
  - export_log_v3.json
